# Notebook 01 — Source Understanding
## Class 4 Skill: Map business questions → required information → source systems

This notebook covers:
- Identifying the business problem and the decision it needs to support
- Mapping each business question to the information required
- Identifying which source system holds that information
- Understanding grain, ownership, and gaps

> FDE mindset: Before touching any data, we ask *what decision does this need to support* and *where does that information live*.

## 1. Business Context

**Client:** NYC Taxi & Limousine Commission (TLC)

**Business problem:** The TLC Operations Director needs a monthly operational view of fleet performance. Currently there is no single system that answers: which zones generate the most revenue, whether trip durations are normal, or which vendors are supplying reliable data.

**Primary KPI:** Revenue per Zone per Month (fare + tip, validated trips only)

**Decision the output supports:** Zone-level dispatch prioritisation and vendor compliance review.

## 2. Business Question → Information → Source Map

| Business Question | Required Information | Source System | Source Type |
|---|---|---|---|
| Which zones drive the most revenue? | fare_amount, tip_amount, PULocationID | TLC Parquet | Parquet file |
| What is the zone name/borough for each ID? | LocationID → name, borough | TLC Zone CSV | CSV file |
| Are trip durations normal? | pickup_datetime, dropoff_datetime | TLC Parquet | Parquet file |
| Which vendors have data quality issues? | VendorID, trip counts | TLC Parquet + vendor metadata | Parquet + CSV |
| What do the fields mean? | Data dictionary / field definitions | NYC Open Data SODA API | REST API / JSON |

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import pandas as pd
from config import (
    TLC_PARQUET_URL, TLC_ZONE_CSV_URL, SODA_METADATA_URL,
    RAW_PARQUET, RAW_ZONES, RAW_VENDORS
)

print('Source inventory:')
sources = [
    {'ID': 'S1', 'Name': 'TLC Yellow Trip Records (Jan 2024)', 'Type': 'Parquet', 'Grain': '1 row/trip', 'URL': TLC_PARQUET_URL},
    {'ID': 'S2', 'Name': 'TLC Zone Lookup',                   'Type': 'CSV',     'Grain': '1 row/zone', 'URL': TLC_ZONE_CSV_URL},
    {'ID': 'S3', 'Name': 'NYC Open Data Metadata API',        'Type': 'JSON/API','Grain': 'field defs', 'URL': SODA_METADATA_URL},
    {'ID': 'S4', 'Name': 'Vendor Metadata (embedded)',        'Type': 'CSV',     'Grain': '1 row/vendor', 'URL': str(RAW_VENDORS)},
]
df_sources = pd.DataFrame(sources)
df_sources

## 3. Source Grain Analysis

Understanding the grain of each source is critical before joining them.

- **S1 (Trips):** One row per taxi trip. The natural key is `(VendorID, tpep_pickup_datetime)` — not a surrogate key. TLC documentation notes that the same trip can appear in both vendor files in rare cases (known duplication issue).
- **S2 (Zones):** One row per LocationID (265 total zones). This is a slowly-changing dimension — zones rarely change.
- **S3 (Metadata):** One JSON object per dataset field. Used for column-level documentation, not for joining.
- **S4 (Vendors):** One row per VendorID. Only two vendors exist in Yellow Taxi data.

In [ ]:
# Show vendor metadata to confirm grain
import pandas as pd
vendor_meta = pd.DataFrame([
    {'VendorID': 1, 'vendor_name': 'Creative Mobile Technologies (CMT)'},
    {'VendorID': 2, 'vendor_name': 'VeriFone Inc.'},
])
print('Vendor dimension — grain: 1 row per vendor')
print(vendor_meta.to_string(index=False))

## 4. Important Gaps Identified

| Gap | Impact | Handling Decision |
|---|---|---|
| No route geometry — only origin/dest zones | Speed is approximated, not exact | Flagged in M4 metric; documented as Limitation |
| Tip for cash payments = NULL (systematic) | Underestimates revenue for cash trips | Known issue; included in KUAL section |
| No driver or vehicle ID | Can't attribute behaviour to individual driver | Out of scope |
| LocationID 264, 265 = "Unknown" | Some trips have no borough attribution | Included with UNKNOWN_ZONE flag |
| No real-time feed | Pipeline is trailing-month only | Acceptable for monthly KPI use case |

> **FDE note:** These gaps were discovered by reading the TLC data dictionary and cross-checking against the SODA API metadata, not by looking at the data first.

## 5. Source Ownership & Access

| Source | Owner | Access | Rate Limit / SLA |
|---|---|---|---|
| S1, S2 | TLC | Public HTTP download (no auth) | No rate limit on static files |
| S3 | NYC Open Data | SODA API (no API key) | 1000 req/day unauthenticated |
| S4 | TLC docs (embedded) | Local file in repo | N/A |

**Access risk:** The TLC CDN (CloudFront) is highly reliable but outside our control. The pipeline preserves raw files on disk so that a CDN outage does not block a re-run.